# Missing values

## Loading the data

**Preprocessing** data is one of the most important tasks in Machine Learning. If the data is not well prepared, *Machine Learning* algorithms won't work correctly. We use the California housing dataset, separating the **predictors** from the **target variable** (the **labels**), since we won't necessarily apply the same **transformations** to both.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

housing = pd.read_csv("../data/housing.csv")
train_set, test_set = train_test_split(
    housing, test_size=0.2, random_state=42,
    stratify=pd.cut(housing["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5]))

X_train = train_set.drop("median_house_value", axis=1)  # features
y_train = train_set["median_house_value"].copy()        # target (labels)

X_train.head().T

,12655,15502,2908,14053,20496
longitude,-121.46,-117.23,-119.04,-117.13,-118.7
latitude,38.52,33.09,35.37,32.75,34.28
housing_median_age,29.0,7.0,44.0,24.0,27.0
total_rooms,3873.0,5320.0,1618.0,1877.0,3536.0
total_bedrooms,797.0,855.0,310.0,519.0,646.0
population,2237.0,2015.0,667.0,898.0,1837.0
households,706.0,768.0,300.0,483.0,580.0
median_income,2.1736,6.3373,2.875,2.2264,4.4964
ocean_proximity,INLAND,NEAR OCEAN,INLAND,NEAR OCEAN,<1H OCEAN


## Identification of unavailable values

The `total_bedrooms` column has unavailable values. Normally we'll speak of unavailable values, **missing values**, **null** or **na** (not available) as synonyms, although we need to be careful about how those values were collected, since if there are two types of values (null and empty *string*, for example) there could be implicit information.

> **Terminology note:** In data science, several terms are used somewhat interchangeably:
> - **Missing value**: The general concept—a value that should exist but doesn't.
> - **NA (Not Available)**: A common label for missing values, used in R and pandas (`pd.NA`).
> - **NaN (Not a Number)**: Originally from IEEE 754 floating-point standard, used in NumPy/pandas for missing floats.
> - **Null**: Common in databases (SQL NULL); in pandas, `None` for object columns.
> - **None**: Python value
>
> In pandas, `isna()` and `isnull()` are aliases—both detect NA and NaN values. The important distinction is between data that is *structurally* missing (not recorded) vs. *semantically* missing (e.g., "not applicable")—these may require different handling strategies.

In [ ]:
X_train.isna().sum()  # the isnull() method is an alias for isna()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        158
population              0
households              0
median_income           0
ocean_proximity         0
dtype: int64

In [ ]:
null_rows_idx = X_train.isnull().any(axis=1)  # boolean mask of rows with null values
X_train.loc[null_rows_idx].head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
1606,-122.08,37.88,26.0,2947.0,NaN,825.0,626.0,2.9330,NEAR BAY
10915,-117.87,33.73,45.0,2264.0,NaN,1970.0,499.0,3.4193,<1H OCEAN
19150,-122.70,38.35,14.0,2313.0,NaN,954.0,397.0,3.7813,<1H OCEAN
4186,-118.23,34.13,48.0,1308.0,NaN,835.0,294.0,4.2891,<1H OCEAN
16885,-122.40,37.58,26.0,3281.0,NaN,1145.0,480.0,6.3580,NEAR OCEAN


## The mechanics of missing data

In data science, we don't just ask *what* is missing; we ask *why* it is missing. This "why" is called the missingness mechanism. Understanding it prevents you from drawing conclusions that are statistically "hallucinated."

### Missing Completely at Random (MCAR)

Under MCAR, the missingness is a "white noise" process. The probability of a data point being missing is a constant, independent of any values in your dataset—whether observed or unobserved.

* **The Logic:** Your observed data is a perfect, smaller reflection of the whole.
* **Formalism:** $P(\text{Missingness} \mid \text{Everything}) = P(\text{Missingness})$
* **On Imputation:** While you *can* delete these rows without biasing your mean, **imputation** is often preferred. Why? Because deleting rows wastes the valid data in other columns, reducing your statistical power. Imputing (e.g., with the mean) helps keep your dataset "whole," though it can artificially deflate your variance.
* **Example:** A sensor loses power for 5 minutes because of a flicker in the local power grid. The temperature it was supposed to record had no influence on the power flicker.

### Missing at Random (MAR)

MAR is the most confusing term in statistics because it sounds like it means "random," but it actually means **systematic—but explainable.**

* **The Confusion:** When we say "at random," we don't mean the missingness is a coin flip. We mean that **once you account for the data you DID see, the remaining uncertainty is random.**
* **The Logic:** The "cause" of the gap is recorded in another column ($X$). If you look at the whole dataset, there is a clear pattern. But if you zoom in on a specific subgroup (e.g., just the women), the pattern disappears and the missingness looks like a pure accident.
* **Formalism:** $P(\text{Missingness} \mid X, Y) = P(\text{Missingness} \mid X)$. This means: "If I know $X$, $Y$ tells me nothing new about why data is missing."
* **Example:** In a health survey, women ($X$) are statistically less likely to report their weight ($Y$) than men. However, within the group of women, the decision to skip the question isn't driven by their *actual* weight—it's just a demographic tendency. If we know the respondent is a woman, we have captured the "cause" of the missing data.
* **Solution:** We use the observed variables ($X$) to "fill in" the gaps in ($Y$).

### Missing Not at Random (MNAR)

In MNAR, the probability of missingness depends on **unobserved information.** This information might be the missing value itself, or a variable you simply failed to measure.

* **Where is the Information?** The distinction between MAR and MNAR is about the **location of the cause**:
    * **MAR:** The cause is in your dataset (in another column).
    * **MNAR:** The cause is **outside** your dataset (either in the missing value itself or in an unobserved variable you didn't record).
* **Example:** People with very high debt are less likely to report their debt levels on a loan application. Or, a thermometer that simply cannot record temperatures above 100°C and returns a "Null" instead.
* **Implication:** This is the most "dangerous" type. Standard imputation fails because the observed data doesn't contain the information needed to explain the gap. You require specialized modeling (like Heckman correction) or domain-specific sensitivity analysis.

### Summary table

| Type | Information Source for Missingness | Strategy |
| :--- | :--- | :--- |
| **MCAR** | None (Noise) | Impute or Delete |
| **MAR** | Other observed variables | Multiple Imputation (using $X$ to predict $Y$) |
| **MNAR** | The missing value itself | Specialized Modeling / Selection Models |

## Deletion of rows with null values (*listwise deletion*)

We can simply delete those incomplete instances, although this is problematic because we're eliminating information. Especially if there are many predictors (since to solve the problem of certain nulls we're losing the information from the other columns).

In [ ]:
X_train_ld_tb = X_train.dropna(subset=["total_bedrooms"])
X_train_ld_tb.loc[null_rows_idx].head()  # verify that rows with null values have been removed

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity


We could also directly delete any row that has a null value in any column:

In [ ]:
X_train_ld_any = X_train.dropna(axis=0)  # remove rows with null values in any column
X_train_ld_any.loc[null_rows_idx].head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity


## Deletion of the entire column

Deleting the entire column is an option if it's not an important variable, but in this case it seems to be important given that, although that *feature* is not the one that correlates most directly with the target variable, it's one of the two used to calculate `bedrooms_ratio`, which is the second most correlated (see [Feature engineering](feature_engineering.ipynb)).

In [ ]:
X_train_drop_tb = X_train.drop(columns="total_bedrooms")
X_train_drop_tb.loc[null_rows_idx].head()

,longitude,latitude,housing_median_age,total_rooms,population,households,median_income,ocean_proximity
1606,-122.08,37.88,26.0,2947.0,825.0,626.0,2.9330,NEAR BAY
10915,-117.87,33.73,45.0,2264.0,1970.0,499.0,3.4193,<1H OCEAN
19150,-122.70,38.35,14.0,2313.0,954.0,397.0,3.7813,<1H OCEAN
4186,-118.23,34.13,48.0,1308.0,835.0,294.0,4.2891,<1H OCEAN
16885,-122.40,37.58,26.0,3281.0,1145.0,480.0,6.3580,NEAR OCEAN


The rows are still there in this case because the null mask was computed before. If we search for nulls now in `X_train_drop_tb`, we won't find any.

In [ ]:
bool(X_train_drop_tb.isnull().any(axis=None))  # verify that there are no null values left

False

We could also directly delete all columns with nulls:

In [ ]:
bool(X_train.dropna(axis=1).isnull().any(axis=None))

False

## Imputation of a value

**Imputation** of a certain value (such as zero, the mean or the median) to those unavailable fields is an option if we believe that unavailable values don't respond to any specific cause, and don't bias the variable's distribution.

In the California dataset, `total_bedrooms` has only 207 missing values (~1%) out of 20,640 records (a small fraction). Without additional information about *why* these values are missing, median imputation is a reasonable choice, as it's robust to potential outliers.

Imputation of the mean (***mean***) is more sensitive to **outliers**, since an extreme value can greatly affect the mean (see [Outliers and capped values](outliers.ipynb)). The median (***median***) is more robust to extreme values. The mode (***mode***) is the value that repeats most, and is useful for categorical variables, but not as much for continuous variables.

[<img src="../img/mean_outliers.jpg" width="300">](https://www.kaggle.com/code/nareshbhat/outlier-the-silent-killer)

In [ ]:
median = X_train["total_bedrooms"].median()
total_bedrooms_filled = X_train["total_bedrooms"].fillna(median)
total_bedrooms_filled.loc[null_rows_idx].head()

1606     433.0
10915    433.0
19150    433.0
4186     433.0
16885    433.0
Name: total_bedrooms, dtype: float64

Now all these rows have in `total_bedrooms` the median value of `total_bedrooms`.

### `SimpleImputer`

The `SimpleImputer` class from scikit-learn allows us to do this more easily. We create an instance of `SimpleImputer` indicating that we want to impute null values with the median, and then use the `fit()` method to calculate the median of each column and the `transform()` method to apply the imputation to all columns (see [Types of estimators](sklearn_estimators.ipynb)).

Let's see how this method would be applied to all numerical fields in the dataframe (remember that `ocean_proximity` is categorical -text values-, and we can't calculate the median of text).

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

In [ ]:
housing_num = X_train.select_dtypes(include=[np.number])  # select numerical columns

In [ ]:
imputer.fit(housing_num)  # calculate the median of each numerical column
imputer.statistics_  # median of each numerical column

array([-118.51   ,   34.26   ,   29.     , 2119.     ,  433.     ,
       1164.     ,  408.     ,    3.54155])

We can verify that the values are the same as those calculated by the dataframe's `median()` method.

In [ ]:
housing_num.median().values

array([-118.51   ,   34.26   ,   29.     , 2119.     ,  433.     ,
       1164.     ,  408.     ,    3.54155])

In [ ]:
housing_num_array_tr = imputer.transform(housing_num)  # replace null values with the median
housing_num_array_tr

array([[-1.2146e+02,  3.8520e+01,  2.9000e+01, ...,  2.2370e+03,
         7.0600e+02,  2.1736e+00],
       [-1.1723e+02,  3.3090e+01,  7.0000e+00, ...,  2.0150e+03,
         7.6800e+02,  6.3373e+00],
       [-1.1904e+02,  3.5370e+01,  4.4000e+01, ...,  6.6700e+02,
         3.0000e+02,  2.8750e+00],
       ...,
       [-1.2272e+02,  3.8440e+01,  4.8000e+01, ...,  4.5800e+02,
         1.7200e+02,  3.1797e+00],
       [-1.2270e+02,  3.8310e+01,  1.4000e+01, ...,  1.2080e+03,
         5.0100e+02,  4.1964e+00],
       [-1.2214e+02,  3.9970e+01,  2.7000e+01, ...,  6.2500e+02,
         1.9700e+02,  3.1319e+00]], shape=(16512, 8))

`transform()` returns a NumPy array, but we could convert it back to a Pandas DataFrame.

In [ ]:
housing_tr = pd.DataFrame(housing_num_array_tr, columns=housing_num.columns, index=housing_num.index)
housing_tr.loc[null_rows_idx].head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
1606,-122.08,37.88,26.0,2947.0,433.0,825.0,626.0,2.9330
10915,-117.87,33.73,45.0,2264.0,433.0,1970.0,499.0,3.4193
19150,-122.70,38.35,14.0,2313.0,433.0,954.0,397.0,3.7813
4186,-118.23,34.13,48.0,1308.0,433.0,835.0,294.0,4.2891
16885,-122.40,37.58,26.0,3281.0,433.0,1145.0,480.0,6.3580


We could also directly use the `fit_transform()` method of `SimpleImputer` to calculate the value to impute (with `fit()`) and apply it (with `transform()`) in a single step.

And we could also use the `.set_output(transform="pandas")` method of the imputer so that the result is a Pandas DataFrame.

Therefore, the entire process detailed above can be summarized in a single line of code:

In [ ]:
housing_tr = SimpleImputer(strategy="median").set_output(transform="pandas").fit_transform(housing_num)
housing_tr.loc[null_rows_idx].head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
1606,-122.08,37.88,26.0,2947.0,433.0,825.0,626.0,2.9330
10915,-117.87,33.73,45.0,2264.0,433.0,1970.0,499.0,3.4193
19150,-122.70,38.35,14.0,2313.0,433.0,954.0,397.0,3.7813
4186,-118.23,34.13,48.0,1308.0,433.0,835.0,294.0,4.2891
16885,-122.40,37.58,26.0,3281.0,433.0,1145.0,480.0,6.3580


### Keeping track of what was imputed

With `add_indicator=True`, `SimpleImputer` appends a binary column flagging the rows where a value was imputed. This lets a model learn whether the *fact* of being missing is informative, which is useful when the mechanism is MAR or MNAR rather than MCAR.

In [ ]:
imputer_with_flag = SimpleImputer(strategy="median", add_indicator=True).set_output(transform="pandas")
imputer_with_flag.fit_transform(housing_num).loc[null_rows_idx].head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,missingindicator_total_bedrooms
1606,-122.08,37.88,26.0,2947.0,433.0,825.0,626.0,2.9330,1.0
10915,-117.87,33.73,45.0,2264.0,433.0,1970.0,499.0,3.4193,1.0
19150,-122.70,38.35,14.0,2313.0,433.0,954.0,397.0,3.7813,1.0
4186,-118.23,34.13,48.0,1308.0,433.0,835.0,294.0,4.2891,1.0
16885,-122.40,37.58,26.0,3281.0,433.0,1145.0,480.0,6.3580,1.0


## Predictive models to impute values

There are more advanced methods such as using **prediction models** (treating the column with null values as the target variable and the rest of the columns as *features*). For example, the **K-Nearest Neighbors (KNN)** algorithm can be used to predict the null values of `total_bedrooms` from the most similar records. Scikit-Learn has a `KNNImputer` class that does this.

Since KNN is based on distances, two things are needed before applying it: the categorical column must be encoded as numbers (see [Categorical encoding](categorical_encoding.ipynb)) and all columns must be scaled, otherwise the features with the largest magnitudes would dominate the distances (see [Feature scaling](feature_scaling.ipynb)). A [pipeline](pipelines.ipynb) chains the three steps:

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

knn_imputation = make_pipeline(
    make_column_transformer(
        (OneHotEncoder(sparse_output=False), ["ocean_proximity"]),  # encode the categorical column
        remainder="passthrough",  # the numerical columns pass unchanged
        verbose_feature_names_out=False),
    StandardScaler(),  # every column on a comparable scale, so no one dominates the distances
    KNNImputer(n_neighbors=5),  # fill each gap with the mean of its 5 nearest complete neighbors
).set_output(transform="pandas")

X_train_knn_imputed = knn_imputation.fit_transform(X_train)
bool(X_train_knn_imputed.isna().any(axis=None))  # verify that there are no null values left

False

In [ ]:
X_train_knn_imputed.loc[null_rows_idx, ["total_bedrooms", "total_rooms", "households"]].head()

,total_bedrooms,total_rooms,households
1606,0.208130,0.151734,0.343342
10915,0.004084,-0.167671,0.005292
19150,-0.250852,-0.144756,-0.266212
4186,-0.593028,-0.614744,-0.540378
16885,-0.132108,0.307929,-0.045282


The imputed `total_bedrooms` values are on the standardized scale because the scaler runs before the imputer, and each one is different: it depends on the neighbors of that district rather than on a single global statistic. The price is computational: KNN has to compare every incomplete row with the whole training set.